# Antahkarana v11 — IEEE Evaluation Pipeline

**Hardware**: NVIDIA L4 24GB · 32 vCPUs · 128GB RAM  
**Engine**: vLLM continuous batching · BFloat16  
**Model**: Meta-Llama-3.1-8B-Instruct

Run cells in order. Cell 1 installs, Cell 2 sets paths, Cell 3 onwards executes.

In [ ]:
# CELL 1 — Install dependencies
import subprocess, sys

def pip(pkg, label):
    print(f'Installing {label}...', end=' ')
    r = subprocess.run(
        f'{sys.executable} -m pip install {pkg} -q',
        shell=True, capture_output=True, text=True, timeout=600
    )
    print('✓' if r.returncode == 0 else f'✗ {r.stderr[-200:]}')

pip('vllm',                                      'vLLM')
pip('datasets transformers accelerate',           'HuggingFace stack')
pip('scipy matplotlib numpy',                     'scientific stack')
pip('sentence-transformers',                      'sentence-transformers')
print('\n✓ All dependencies installed')

In [ ]:
# CELL 2 — Environment & path setup
import os, sys

# ⚠ Set your HuggingFace token if using gated models
os.environ['HF_TOKEN'] = 'hf_YOUR_TOKEN_HERE'

# Add project root to path
PROJECT_ROOT = os.path.abspath('.')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# GPU check
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

import subprocess
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,utilization.gpu',
                    '--format=csv,noheader'], capture_output=True, text=True)
print('\nnvidia-smi:', r.stdout.strip())

In [ ]:
# CELL 3 — Load vLLM engine (runs ONCE; takes ~60s)
import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s — %(message)s')

from vllm_engine import get_engine
engine = get_engine()
print(f'\n✓ Engine ready (loaded in {engine._load_time:.1f}s)')

In [ ]:
# CELL 4 — Load all datasets
from datasets import load_all_datasets

N_MAIN = 100
datasets = load_all_datasets(n=N_MAIN)

for name, data in datasets.items():
    status = '✓' if data else '✗'
    print(f'  {status} {name}: {len(data)} samples')

In [ ]:
# CELL 5 — Run all baselines (batched via vLLM)
import time
from baselines import run_all_baselines

all_results = {}
baselines = ['direct', 'cot', 'self_consistency', 'tot']

for ds_name, samples in datasets.items():
    if not samples:
        continue
    print(f'\n▶ {ds_name} baselines...')
    t0 = time.time()
    br = run_all_baselines(samples, ds_name, methods=baselines)
    print(f'  done in {time.time()-t0:.1f}s')
    all_results.setdefault(ds_name, {}).update(br)

print('\n✓ Baselines complete')

In [ ]:
# CELL 6 — Run Antahkarana
from antahkarana import AntahkaranaSystem

antahkarana = AntahkaranaSystem()

for ds_name, samples in datasets.items():
    if not samples:
        continue
    print(f'\n▶ Antahkarana on {ds_name} ({len(samples)} samples)...')
    t0 = time.time()
    ant_res = antahkarana.run_batch(samples, ds_name)
    elapsed = time.time() - t0
    all_results.setdefault(ds_name, {})['antahkarana'] = ant_res
    print(f'  done in {elapsed:.1f}s ({len(samples)/elapsed:.1f} samp/s)')

print('\n✓ Antahkarana complete')

In [ ]:
# CELL 7 — Compute metrics
from evaluation import score_result, aggregate_scores, compute_throughput_stats
import json

summary = {}

for ds_name, method_results in all_results.items():
    samples    = datasets.get(ds_name, [])
    sample_map = {s.get('id', i): s for i, s in enumerate(samples)}
    summary[ds_name] = {}

    for method, results in method_results.items():
        scores_list = []
        for i, r in enumerate(results):
            sample = sample_map.get(r.get('id'), samples[i] if i < len(samples) else {})
            sc = score_result(r, sample, ds_name)
            scores_list.append(sc)
            r['scores'] = sc

        agg = aggregate_scores(scores_list)
        thr = compute_throughput_stats(results)
        agg.update({f'throughput_{k}': {'mean': v} for k, v in thr.items()})
        summary[ds_name][method] = agg

        em = agg.get('em', {}).get('mean', 0)
        f1 = agg.get('f1', {}).get('mean', 0)
        print(f'  [{ds_name}][{method}] EM={em:.3f} F1={f1:.3f}')

# Save
with open('results/processed/metrics_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print('\n✓ Metrics saved')

In [ ]:
# CELL 8 — Ablation study
from evaluation import run_ablation_study

N_ABLATION = 50
hotpot_abl = datasets.get('hotpotqa', [])[:N_ABLATION]

ablation_raw     = run_ablation_study(hotpot_abl, 'hotpotqa')
ablation_summary = {}

sample_map_abl = {s.get('id', i): s for i, s in enumerate(hotpot_abl)}
for cfg_name, results in ablation_raw.items():
    scores_list = []
    for i, r in enumerate(results):
        sample = sample_map_abl.get(r.get('id'), hotpot_abl[i] if i < len(hotpot_abl) else {})
        scores_list.append(score_result(r, sample, 'hotpotqa'))
    agg = aggregate_scores(scores_list)
    ablation_summary[cfg_name] = agg
    print(f'  [{cfg_name}] EM={agg.get("em",{}).get("mean",0):.3f} F1={agg.get("f1",{}).get("mean",0):.3f}')

with open('results/ablation/ablation_summary.json', 'w') as f:
    json.dump(ablation_summary, f, indent=2, default=str)
print('\n✓ Ablation complete')

In [ ]:
# CELL 9 — Statistical significance
from evaluation import run_significance_tests

sig_results = {}
for ds_name, method_results in all_results.items():
    ant_res = method_results.get('antahkarana', [])
    ant_f1  = [r.get('scores', {}).get('f1', 0.0) for r in ant_res]

    baseline_f1 = {}
    for m in ('direct', 'cot', 'self_consistency', 'tot'):
        res = method_results.get(m, [])
        baseline_f1[m] = [r.get('scores', {}).get('f1', 0.0) for r in res]

    sig = run_significance_tests(ant_f1, baseline_f1)
    sig_results[ds_name] = sig

    print(f'\n[{ds_name}]')
    for method, s in sig.items():
        print(f'  vs {method:20s}: Δ{s["improvement_pct"]:+.1f}% F1  '
              f'p={s["p_value"]:.4f} {s["significance"]}')

with open('results/stats/significance.json', 'w') as f:
    json.dump(sig_results, f, indent=2)
print('\n✓ Significance tests complete')

In [ ]:
# CELL 10 — Generate plots
from evaluation import generate_all_plots

plot_paths = generate_all_plots(summary, ablation_summary)
print(f'\n✓ Generated {len(plot_paths)} plots:')
for p in plot_paths:
    print(f'  {p}')

# Display inline
from IPython.display import Image, display
for p in plot_paths:
    display(Image(p))

In [ ]:
# CELL 11 — Save CSV + final report
import csv

# Metrics CSV
csv_rows = []
for ds, methods in summary.items():
    for method, agg in methods.items():
        csv_rows.append({
            'dataset':  ds, 'method': method,
            'em_mean':  agg.get('em',{}).get('mean',0),
            'em_std':   agg.get('em',{}).get('std',0),
            'f1_mean':  agg.get('f1',{}).get('mean',0),
            'f1_std':   agg.get('f1',{}).get('std',0),
            'sf_em':    agg.get('sf_em',{}).get('mean','N/A'),
            'sf_f1':    agg.get('sf_f1',{}).get('mean','N/A'),
            'latency_s':   agg.get('throughput_mean_latency_s',{}).get('mean',0),
            'throughput':  agg.get('throughput_throughput_sps',{}).get('mean',0),
        })

with open('results/processed/metrics_table.csv','w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=csv_rows[0].keys())
    w.writeheader(); w.writerows(csv_rows)

# Print final summary
all_imps = [
    s['improvement_pct']
    for ds_comps in sig_results.values()
    for s in ds_comps.values()
    if s['significance'] != 'ns'
]
if all_imps:
    print(f'\n{'='*60}')
    print(f'CONCLUSION: Antahkarana improves F1 by {sum(all_imps)/len(all_imps):.1f}%'
          f' on average (max {max(all_imps):.1f}%) over baselines.')
    print(f'{'='*60}')

print('\n✓ All outputs saved to results/')